# Supervised Tabular Training

This notebook trains a compact flat classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run. Use it as a tabular comparison point, not as the main nested-data story.

Import the runtime pieces used in the full training loop: Lightning for optimization and Polars for reading the bundled JSONL records.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


The Wine buffer is flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemistry fields to predict the cultivar.


In [2]:
records = pl.read_ndjson("docs/data/wine.jsonl").head(48)

records.head()


alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,cultivar
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
14.23,1.71,2.43,15.6,127.0,2.8,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,"""class_0"""
12.37,0.94,1.36,10.6,88.0,1.98,0.57,0.28,0.42,1.95,1.05,1.82,520.0,"""class_1"""
12.86,1.35,2.32,18.0,122.0,1.51,1.25,0.21,0.94,4.1,0.76,1.29,630.0,"""class_2"""
13.2,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.4,1050.0,"""class_0"""
12.33,1.1,2.28,16.0,101.0,2.05,1.09,0.63,0.41,3.27,1.25,1.67,680.0,"""class_1"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `embed=True` asks the root node to return embeddings after training.


In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol"),
    j2v.Number("malic_acid"),
    j2v.Number("color_intensity"),
    j2v.Number("proline"),
    j2v.Category("cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)


`PolarsDataModule.from_model(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

The plot confirms that the target and root embedding are both configured before inference.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  26,039                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  5                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

After training, `predict` returns typed outputs for supervised targets and `embed` returns vectors from the nodes configured with `embed=True`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))
pprint(model.embed(batch))

{
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.3911939859390259, 0.3975856602191925, 0.3878841698169708],
│   │   │   'null': [0.10189476609230042, 0.09963122755289078, 0.10302100330591202],
│   │   │   'padded': [0.18265023827552795, 0.18005774915218353, 0.18428544700145721],
│   │   │   'masked': [0.19199138879776, 0.19875948131084442, 0.19663174450397491],
│   │   │   'other': [0.13226954638957977, 0.12396583706140518, 0.12817762792110443]
│   │   },
│   │   'content': {
│   │   │   'value': ['class_2', 'class_2', 'class_2'],
│   │   │   'probability': [0.4304247796535492, 0.45131179690361023, 0.43881314992904663],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.4304247796535492},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.32945936918258667}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.45131179690361023},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.3098171651363373}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_2', 'probability': 0.43881314992904663},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.3301960229873657}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.12915396690368652,
│   │   │   │   -0.424543559551239,
│   │   │   │   0.11516954004764557,
│   │   │   │   -0.4780794084072113,
│   │   │   │   0.23538409173488617,
│   │   │   │   0.006434302311390638,
│   │   │   │   0.28101304173469543,
│   │   │   │   0.18555231392383575,
│   │   │   │   -0.25864294171333313,
│   │   │   │   0.03454628959298134,
│   │   │   │   -0.1052267998456955,
│   │   │   │   -0.384408175945282,
│   │   │   │   0.24591100215911865,
│   │   │   │   0.24900420010089874,
│   │   │   │   0.20631590485572815,
│   │   │   │   -0.020965296775102615
│   │   │   ],
│   │   │   [
│   │   │   │   0.03474922105669975,
│   │   │   │   -0.4085697531700134,
│   │   │   │   0.16862007975578308,
│   │   │   │   -0.44963106513023376,
│   │   │   │   0.27305009961128235,
│   │   │   │   0.06149492785334587,
│   │   │   │   0.3247949182987213,
│   │   │   │   0.20823360979557037,
│   │   │   │   -0.22624541819095612,
│   │   │   │   0.016177326440811157,
│   │   │   │   -0.10928370803594589,
│   │   │   │   -0.41611936688423157,
│   │   │   │   0.17831985652446747,
│   │   │   │   0.2638566792011261,
│   │   │   │   0.1737714409828186,
│   │   │   │   -0.07687520235776901
│   │   │   ],
│   │   │   [
│   │   │   │   0.06450562179088593,
│   │   │   │   -0.419446736574173,
│   │   │   │   0.16716282069683075,
│   │   │   │   -0.4560895562171936,
│   │   │   │   0.25149986147880554,
│   │   │   │   0.031172089278697968,
│   │   │   │   0.3083977997303009,
│   │   │   │   0.19535231590270996,
│   │   │   │   -0.24731571972370148,
│   │   │   │   0.03132228925824165,
│   │   │   │   -0.11089344322681427,
│   │   │   │   -0.4004902243614197,
│   │   │   │   0.19413115084171295,
│   │   │   │   0.27138039469718933,
│   │   │   │   0.19261695444583893,
│   │   │   │   -0.05634898319840431
│   │   │   ]
│   │   ]
│   }
}